# Azure AI Foundry – Photo Upload & Structured Output with Agents

**Goal:** Upload a photo to Azure Blob Storage, then use an **Azure AI Foundry Agent** backed by GPT-4o (vision) to analyse the image and return fully structured JSON via tool/function calling.

---

## How to use this notebook

1. Work through **Setup Steps 1–6** below in the Azure portal / Azure AI Foundry portal (markdown cells – no code to run).
2. Copy the values you collect (connection string, storage key, etc.) into **Section 7 – Configuration**.
3. Run all code cells in order (Sections 8 onwards).


---
## Setup Step 1 – Create a Resource Group

All Azure resources must live in the same region. We recommend **East US** or **Sweden Central** as both have GPT-4o available.

1. Go to the [Azure Portal](https://portal.azure.com).
2. Search for **Resource groups** → click **+ Create**.
3. Fill in:
   - **Subscription** – select your subscription
   - **Resource group name** – e.g. `rg-aifoundry-workshop`
   - **Region** – e.g. `East US`
4. Click **Review + create** → **Create**.

> ✅ **Collect:** resource group name and region – you will need them in the next steps.


---
## Setup Step 2 – Deploy an Azure OpenAI Service with GPT-4o

1. In the Azure Portal, click **+ Create a resource** → search **Azure OpenAI** → **Create**.
2. Fill in:
   - **Resource group** – the one created in Step 1
   - **Name** – e.g. `oai-foundry-workshop`
   - **Region** – same region as above
   - **Pricing tier** – `Standard S0`
3. Click **Review + create** → **Create**. Wait for deployment to complete.
4. Once deployed, go to the resource → **Keys and Endpoint**.
   - Copy **KEY 1** → save as `OAI_KEY`
   - Copy **Endpoint** → save as `OAI_ENDPOINT` (format: `https://<name>.openai.azure.com/`)
5. In the same resource, click **Model deployments** → **Manage Deployments** → opens **Azure OpenAI Studio**.
6. Click **+ Deploy model** → **Deploy base model** → search `gpt-4o` → select it → click **Confirm**.
   - **Deployment name** – e.g. `gpt-4o` (note this down as `VISION_DEPLOYMENT`)
   - Leave all other settings as default → **Deploy**.

> ✅ **Collect:** `OAI_KEY`, `OAI_ENDPOINT`, `VISION_DEPLOYMENT`


---
## Setup Step 3 – Create an Azure AI Foundry Hub and Project

### 3a – Create the Hub

1. Go to [ai.azure.com](https://ai.azure.com) and sign in.
2. Click **All hubs + projects** (left sidebar) → **+ New hub**.
3. Fill in:
   - **Hub name** – e.g. `aif-hub-workshop`
   - **Subscription** and **Resource group** – same as Step 1
   - **Location** – same region
   - Leave **Storage account**, **Key Vault**, and **Container Registry** as *Create new* (defaults are fine)
4. Click **Next** → **Create**. Hub provisioning takes ~2 minutes.

### 3b – Connect Your Azure OpenAI Resource to the Hub

1. Open the Hub you just created → click **Settings** → **Connected resources** → **+ New connection**.
2. Select **Azure OpenAI**.
3. Pick the OpenAI resource from Step 2 → click **Add connection**.

### 3c – Create a Project inside the Hub

1. From the Hub page click **+ New project**.
2. Fill in:
   - **Project name** – e.g. `aif-project-workshop`
3. Click **Create**. Project is created in seconds.

### 3d – Copy the Project Connection String

1. Open the project → **Overview** tab.
2. Find **Project connection string** (right panel). It looks like:
   ```
   eastus.api.azureml.ms;00000000-0000-0000-0000-000000000000;rg-aifoundry-workshop;aif-project-workshop
   ```
3. Copy it → save as `PROJECT_CONNECTION_STRING`.

> ✅ **Collect:** `PROJECT_CONNECTION_STRING`


---
## Setup Step 4 – Create Azure Blob Storage for Photos

1. In the Azure Portal, click **+ Create a resource** → search **Storage account** → **Create**.
2. Fill in:
   - **Resource group** – same as Step 1
   - **Storage account name** – e.g. `stphotosworkshop` (must be globally unique, 3–24 lowercase chars)
   - **Region** – same region
   - **Redundancy** – `Locally-redundant storage (LRS)` (cheapest for a workshop)
3. Click **Review** → **Create**.
4. Once deployed, open the storage account → **Security + networking** → **Access keys**.
   - Click **Show** next to **key1** → copy the key → save as `STORAGE_ACCOUNT_KEY`.
   - Copy the **Storage account name** → save as `STORAGE_ACCOUNT_NAME`.
5. In the same storage account, go to **Data storage** → **Containers** → **+ Container**.
   - **Name:** `photos`
   - **Public access level:** Private
   - Click **Create**.

> ✅ **Collect:** `STORAGE_ACCOUNT_NAME`, `STORAGE_ACCOUNT_KEY`


---
## Setup Step 5 – Assign Roles (for Managed Identity in Microsoft Fabric)

When running this notebook inside **Microsoft Fabric**, authentication uses a **Managed Identity** automatically via `DefaultAzureCredential`. You need to grant it access.

> **Skip this step** if you plan to authenticate with a personal account (interactive login) or a service principal secret instead.

### Grant the Fabric Managed Identity access to Azure AI Foundry

1. In the Azure Portal, open your **AI Foundry Project** resource (under *Machine Learning* resource type).
2. Go to **Access control (IAM)** → **+ Add** → **Add role assignment**.
3. Search for and select the role **Azure AI Developer** → click **Next**.
4. Under **Assign access to** choose **Managed identity**.
5. Click **+ Select members** → find your Fabric workspace's managed identity → **Select** → **Review + assign**.

### Grant the Fabric Managed Identity access to Blob Storage

1. Open the **Storage account** from Step 4.
2. Go to **Access control (IAM)** → **+ Add** → **Add role assignment**.
3. Select **Storage Blob Data Contributor** → **Next**.
4. Assign to the same Managed identity as above → **Review + assign**.

> ✅ Roles assigned – Managed Identity can now upload blobs and call AI Foundry Agent Service.


---
## Setup Step 6 – Verify the Environment in Microsoft Fabric

The `azure-ai-projects` package must be available in your Fabric environment.
It is already listed in `environment.yaml` for this workshop.

**If you have not yet attached the workshop environment to your notebook:**
1. Open the notebook in Fabric → click the environment selector in the top toolbar.
2. Choose the environment created from `hands-on/environment.yaml`.
3. If it is not yet created: go to your workspace → **+ New** → **Environment** → import from YAML.

**Quick check (run this cell):**


In [ ]:
import importlib
for pkg in ["azure.ai.projects", "azure.storage.blob", "azure.identity"]:
    found = importlib.util.find_spec(pkg.split(".")[0])
    status = "✅ found" if found else "❌ MISSING – install via environment.yaml"
    print(f"{pkg:35s} {status}")


---
## Section 7 – Configuration

Paste the values you collected in Steps 1–4 into the cell below, then run it.


In [ ]:
# ── Azure AI Foundry ─────────────────────────────────────────────────────
# From Step 3d: Project → Overview → Project connection string
PROJECT_CONNECTION_STRING = "<region>.api.azureml.ms;<subscription-id>;<resource-group>;<project-name>"  # TODO

# From Step 2 – the deployment name you gave to gpt-4o
VISION_DEPLOYMENT = "gpt-4o"  # TODO if you used a different deployment name

# ── Azure Blob Storage ────────────────────────────────────────────────────
# From Step 4
STORAGE_ACCOUNT_NAME = "<your-storage-account-name>"  # TODO
STORAGE_ACCOUNT_KEY  = "<your-storage-account-key>"   # TODO
CONTAINER_NAME       = "photos"  # the container you created in Step 4

# ── Image to analyse ─────────────────────────────────────────────────────
# Path to a local image file. Run the next cell to download a sample if needed.
LOCAL_IMAGE_PATH = "/tmp/sample_photo.jpg"

# ── Validation ───────────────────────────────────────────────────────────
assert PROJECT_CONNECTION_STRING != "<region>.api.azureml.ms;<subscription-id>;<resource-group>;<project-name>", \
    "❌ Set PROJECT_CONNECTION_STRING from the Azure AI Foundry portal (Step 3d)"
assert STORAGE_ACCOUNT_NAME != "<your-storage-account-name>", \
    "❌ Set STORAGE_ACCOUNT_NAME from the Azure portal (Step 4)"
assert STORAGE_ACCOUNT_KEY != "<your-storage-account-key>", \
    "❌ Set STORAGE_ACCOUNT_KEY from the Azure portal (Step 4)"

print("✅ Configuration looks good – proceed to the next cell.")


---
## Section 8 – Download a Sample Image *(optional)*

Skip this cell if you already have an image at `LOCAL_IMAGE_PATH`.


In [ ]:
import urllib.request, os

SAMPLE_URL = (
    "https://upload.wikimedia.org/wikipedia/commons/thumb/"
    "4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png"
)

if not os.path.exists(LOCAL_IMAGE_PATH):
    urllib.request.urlretrieve(SAMPLE_URL, LOCAL_IMAGE_PATH)
    print(f"✅ Sample image downloaded → {LOCAL_IMAGE_PATH}")
else:
    print(f"✅ Using existing image at {LOCAL_IMAGE_PATH}")


---
## Section 9 – Upload the Photo to Azure Blob Storage

Uploads the image and generates a **1-hour SAS URL** for the AI Foundry Agent to fetch.


In [ ]:
from azure.storage.blob import BlobServiceClient, BlobSasPermissions, generate_blob_sas
from datetime import datetime, timezone, timedelta
import os

blob_name = os.path.basename(LOCAL_IMAGE_PATH)
connection_string = (
    f"DefaultEndpointsProtocol=https;"
    f"AccountName={STORAGE_ACCOUNT_NAME};"
    f"AccountKey={STORAGE_ACCOUNT_KEY};"
    f"EndpointSuffix=core.windows.net"
)

blob_service_client = BlobServiceClient.from_connection_string(connection_string)

container_client = blob_service_client.get_container_client(CONTAINER_NAME)
if not container_client.exists():
    container_client.create_container()
    print(f"Container '{CONTAINER_NAME}' created.")

blob_client = blob_service_client.get_blob_client(container=CONTAINER_NAME, blob=blob_name)
with open(LOCAL_IMAGE_PATH, "rb") as data:
    blob_client.upload_blob(data, overwrite=True)
print(f"✅ Uploaded '{blob_name}' to container '{CONTAINER_NAME}'")

sas_token = generate_blob_sas(
    account_name=STORAGE_ACCOUNT_NAME,
    container_name=CONTAINER_NAME,
    blob_name=blob_name,
    account_key=STORAGE_ACCOUNT_KEY,
    permission=BlobSasPermissions(read=True),
    expiry=datetime.now(timezone.utc) + timedelta(hours=1),
)
image_url = (
    f"https://{STORAGE_ACCOUNT_NAME}.blob.core.windows.net/"
    f"{CONTAINER_NAME}/{blob_name}?{sas_token}"
)
print(f"✅ SAS URL generated (valid 1 hour)")


---
## Section 10 – Connect to Your Azure AI Foundry Project

Inside Microsoft Fabric, `DefaultAzureCredential` authenticates via Managed Identity automatically.
Outside Fabric you may be prompted for a browser login.


In [ ]:
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

project_client = AIProjectClient.from_connection_string(
    credential=DefaultAzureCredential(),
    conn_str=PROJECT_CONNECTION_STRING,
)

project_name = PROJECT_CONNECTION_STRING.split(";")[-1]
print(f"✅ Connected to AI Foundry project: {project_name}")


---
## Section 11 – Define the Structured-Output Tool

The `extract_image_details` function tool tells the agent **exactly** which fields to return.
The model is forced to call this function instead of responding with free text,
giving you strongly-typed JSON output every time.

Extend the `properties` dictionary to add any domain-specific fields you need
(e.g. product SKU, defect code, confidence threshold).


In [ ]:
from azure.ai.projects.models import FunctionTool

extract_image_details_def = {
    "name": "extract_image_details",
    "description": (
        "Extract structured details from an image. "
        "Populate every field you can determine from the image; "
        "use null for fields that cannot be determined."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "description":       {"type": "string",  "description": "Concise description of what is shown."},
            "main_subject":      {"type": "string",  "description": "Primary subject or object in the image."},
            "colors":            {"type": "array",   "items": {"type": "string"}, "description": "Dominant colours."},
            "objects_detected":  {"type": "array",   "items": {"type": "string"}, "description": "Objects visible."},
            "scene_type":        {"type": "string",  "enum": ["indoor","outdoor","product","document","person","other"]},
            "text_in_image":     {"type": "string",  "description": "Any visible text, or null."},
            "confidence_score":  {"type": "number",  "description": "Confidence 0.0–1.0."},
        },
        "required": ["description","main_subject","colors","objects_detected","scene_type","confidence_score"],
    },
}

function_tool = FunctionTool(functions=[extract_image_details_def])
print(f"✅ Tool '{extract_image_details_def['name']}' registered with {len(extract_image_details_def['parameters']['properties'])} output fields")


---
## Section 12 – Create the AI Foundry Agent

Provision a short-lived agent backed by GPT-4o with our function tool attached.
The agent is deleted at the end of the notebook (Section 15) to keep things tidy.


In [ ]:
agent = project_client.agents.create_agent(
    model=VISION_DEPLOYMENT,
    name="photo-analyser-agent",
    instructions=(
        "You are a computer-vision assistant. "
        "When given an image URL, analyse the image carefully and "
        "ALWAYS respond by calling the extract_image_details function. "
        "Never reply with plain text – only use the function call."
    ),
    tools=function_tool.definitions,
)

print(f"✅ Agent created  – id: {agent.id}  model: {agent.model}")


---
## Section 13 – Create a Thread and Send the Image

A **Thread** is an isolated conversation context within the Foundry Agent Service.
The user message includes the SAS URL of the uploaded photo.


In [ ]:
thread = project_client.agents.create_thread()
print(f"✅ Thread created – id: {thread.id}")

message = project_client.agents.create_message(
    thread_id=thread.id,
    role="user",
    content=[
        {"type": "text",      "text": "Please analyse this image and extract all structured details using the provided tool."},
        {"type": "image_url", "image_url": {"url": image_url, "detail": "high"}},
    ],
)
print(f"✅ Message sent   – id: {message.id}")


---
## Section 14 – Run the Agent and Parse the Structured Output

`create_and_process_run` starts the agent run and **polls automatically until completion**.
We then inspect the run steps to retrieve the `extract_image_details` tool-call arguments.


In [ ]:
import json

run = project_client.agents.create_and_process_run(
    thread_id=thread.id,
    agent_id=agent.id,
)
print(f"Run status: {run.status}")

if run.status == "failed":
    raise RuntimeError(f"Agent run failed: {run.last_error}")

# Find the function-call tool step and extract its arguments
run_steps = project_client.agents.list_run_steps(thread_id=thread.id, run_id=run.id)

structured_output = None
for step in run_steps:
    if step.type == "tool_calls" and step.step_details and step.step_details.tool_calls:
        for tc in step.step_details.tool_calls:
            if hasattr(tc, "function") and tc.function:
                structured_output = json.loads(tc.function.arguments)
                break
    if structured_output:
        break

if structured_output is None:
    raise ValueError("No tool call found. Check that the model deployment supports vision and that the SAS URL is valid.")

print(f"\n✅ Structured output from extract_image_details:")
print(json.dumps(structured_output, indent=2))


---
## Section 15 – Clean Up the Agent

Delete the agent to free Foundry resources. Your thread history is also removed.


In [ ]:
project_client.agents.delete_agent(agent.id)
print(f"✅ Agent {agent.id} deleted")


---
## Section 16 – Load Results into a Spark DataFrame

Convert the structured output into a **Spark DataFrame** so it can be stored in the Lakehouse,
queried with SQL analytics, or used in downstream Fabric pipelines.


In [ ]:
from pyspark.sql import Row

row = Row(
    blob_name        = blob_name,
    description      = structured_output.get("description"),
    main_subject     = structured_output.get("main_subject"),
    colors           = structured_output.get("colors", []),
    objects_detected = structured_output.get("objects_detected", []),
    scene_type       = structured_output.get("scene_type"),
    text_in_image    = structured_output.get("text_in_image"),
    confidence_score = float(structured_output.get("confidence_score", 0.0)),
)

results_df = spark.createDataFrame([row])
display(results_df)


---
## Section 17 – *(Optional)* Save Results to the Lakehouse

Uncomment the lines below to persist the output as a **Delta table** for reporting or downstream analytics.


In [ ]:
# results_df.write.format("delta").mode("append").saveAsTable("image_analysis_results")
# print("✅ Saved to Lakehouse Delta table: image_analysis_results")
